In [4]:
from importlib import reload
import models

reload(models)
import inspect
import os
import sys
import torch
import pandas as pd
import mlflow
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from datetime import datetime
sys.path.append(os.path.abspath(os.path.join('..')))

from models import HybridModel
from utils import mol_to_graph, MLFlowManager, train_hybrid_model, evaluate_hybrid_model
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') # type: ignore


import logging

logging.basicConfig(
    filename="debug_model.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    filemode="w"
)

logger = logging.getLogger(__name__)

def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

features = [
    'alogp', 'psa', 'hba', 'hbd', 'num_ro5_violations', 'qed_weighted',
    'logP_over_PSA', 'HBA_HBD_ratio', 'HBA_HBD_sum'
]
num_features = len(features)
parquet_path = "parquets/subset_50k_stratified.parquet"
df = pd.read_parquet(parquet_path)
dataset_path = "/home/pkuszn/repos/WSzI/src/notebooks/data/chembl_dataset.pt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists(dataset_path):
    print("Loading existing dataset...")
    dataset = torch.load(dataset_path, weights_only=False)
else:
    print("Converting SMILES to graphs...")
    dataset = Parallel(n_jobs=-1)(
        delayed(mol_to_graph)(
            s, 
            y, 
            row[features].values.astype(float)
        ) 
        for s, y, row in tqdm(
            zip(df['canonical_smiles'], df['pic50'], [row for _, row in df.iterrows()]), 
            total=len(df), 
            desc="Converting"
        )
    )
    dataset = [d for d in dataset if d is not None]
    torch.save(dataset, dataset_path)

train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

batch = next(iter(train_loader))

print(batch.y.shape)
print(batch.y[:5])

model = HybridModel(num_node_features=4, num_extra_features=num_features, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

mf = MLFlowManager(experiment_name="ChEMBL_HybridModel_Scaffold_Split")

now = str(int(datetime.now().timestamp()))
run_name = f"Hybrid_Run_{now}"
print("Starting Training...")
best_r2 = float("-inf")
with mf.start_run(run_name):
    for epoch in range(50):

        loss = train_hybrid_model(
            model,
            train_loader,
            optimizer,
            device,
            epoch
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": loss,
            },
            "checkpoint_latest.pt"
        )

        r2, mae, _, _ = evaluate_hybrid_model(
            model,
            test_loader,
            device
        )

        mlflow.log_metric("train_mse", loss, step=epoch)
        mlflow.log_metric("val_r2", r2, step=epoch)
        mlflow.log_metric("val_mae", mae, step=epoch)

        if r2 > best_r2:
            best_r2 = r2

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "r2": r2,
                    "mae": mae,
                },
                "best_model.pt"
            )

            print(
                f"New best model! "
                f"Epoch={epoch} "
                f"R2={r2:.4f} "
                f"MAE={mae:.4f}"
            )

        print(
            f"Epoch {epoch:02d} | "
            f"Train MSE={loss:.4f} | "
            f"Val R2={r2:.4f} | "
            f"Val MAE={mae:.4f}"
        )


    r2, mae, y_true, y_pred = evaluate_hybrid_model(
        model,
        test_loader,
        device
    )

    mlflow.log_artifact("best_model.pt")
    mlflow.log_artifact("checkpoint_latest.pt")

    model_save_path = f"model_{run_name}_weights.pth"
    torch.save(model.state_dict(), model_save_path)
    print(f"Saved weight to {model_save_path}")
    mlflow.log_artifact(model_save_path)

    mlflow.log_metrics({"test_r2": r2, "test_mae": mae})
    log_regression_plots(y_true, y_pred, "Hybrid_Final")
    print(f"Final Test: R2={r2:.4f}, MAE={mae:.4f}")

Loading existing dataset...
torch.Size([32])
tensor([8.5850, 4.5528, 9.0000, 9.7959, 6.6990])
Starting Training...
Epoch 0 | MSE: 6467777.9425 | MAE: 162.1625 | R2: -1584797.0000
New best model! Epoch=0 R2=-1.1802 MAE=2.2194
Epoch 00 | Train MSE=6467777.9425 | Val R2=-1.1802 | Val MAE=2.2194
Epoch 1 | MSE: 21280.2066 | MAE: 3.4114 | R2: -5213.2842
Epoch 01 | Train MSE=21280.2066 | Val R2=-28508.8672 | Val MAE=5.9480
Epoch 2 | MSE: 25359.0667 | MAE: 3.9350 | R2: -6212.7251
New best model! Epoch=2 R2=-0.4139 MAE=1.9275
Epoch 02 | Train MSE=25359.0667 | Val R2=-0.4139 | Val MAE=1.9275
Epoch 3 | MSE: 69.3459 | MAE: 1.9644 | R2: -15.9918
New best model! Epoch=3 R2=-0.2975 MAE=1.8748
Epoch 03 | Train MSE=69.3459 | Val R2=-0.2975 | Val MAE=1.8748
Epoch 4 | MSE: 12699.7982 | MAE: 2.8439 | R2: -3110.8281
New best model! Epoch=4 R2=-0.1061 MAE=1.7432
Epoch 04 | Train MSE=12699.7982 | Val R2=-0.1061 | Val MAE=1.7432
Epoch 5 | MSE: 23.9741 | MAE: 1.7550 | R2: -4.8744
New best model! Epoch=5 R2=-0.